# NeoOLAF native EventStoryLine layer ablation — one document v1.6.3

This portable v1.6.3 keeps the v1.6 source-centric relation experiment unchanged scientifically, but fixes Windows cache-path failures. Layer-2 cache files now live in a deliberately short OS-temp directory (or `NEOOLAF_EVENTSTORYLINE_CACHE_DIR`), use 24-character hashes, and cache I/O is strictly best-effort: a successful LLM/JSON response can never be discarded because a cache file cannot be written. No file under `src/neoolaf` is modified.


In [1]:
from __future__ import annotations

import json
import os
import sys
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/neoolaf").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the NeoOLAF repository.")


def first_existing_path(env_name: str, candidates: list[Path]) -> Path:
    raw = os.environ.get(env_name, "").strip().strip('"').strip("'")
    if raw:
        path = Path(raw).expanduser().resolve()
        if path.is_file():
            return path
        raise FileNotFoundError(f"{env_name} points to a missing file: {path}")
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Could not find {env_name}. Tried:\n"
        + "\n".join(str(Path(x).expanduser().resolve()) for x in candidates)
    )


def first_existing_dir(env_name: str, candidates: list[Path]) -> Path:
    raw = os.environ.get(env_name, "").strip().strip('"').strip("'")
    if raw:
        path = Path(raw).expanduser().resolve()
        if path.is_dir():
            return path
        raise FileNotFoundError(f"{env_name} points to a missing directory: {path}")
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        f"Could not find {env_name}. Tried:\n"
        + "\n".join(str(Path(x).expanduser().resolve()) for x in candidates)
    )


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def strip_gold(record: dict) -> dict:
    return {k: v for k, v in record.items() if k not in {"entities", "relations"}}


def prepare_eventstoryline_views(
    dataset_jsonl: Path,
    selected_document_id: str,
    input_jsonl: Path,
    gold_jsonl: Path,
    smoke5_input_jsonl: Path,
    smoke5_gold_jsonl: Path,
) -> None:
    selected = None
    smoke5 = []
    with dataset_jsonl.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            if len(smoke5) < 5:
                smoke5.append(record)
            if record.get("document_id") == selected_document_id:
                selected = record
            if selected is not None and len(smoke5) >= 5:
                break

    if selected is None:
        raise KeyError(
            f"Document {selected_document_id!r} was not found in {dataset_jsonl}"
        )
    if len(smoke5) != 5:
        raise RuntimeError(f"Expected at least 5 EventStoryLine records, found {len(smoke5)}")

    write_jsonl(input_jsonl, [strip_gold(selected)])
    write_jsonl(gold_jsonl, [selected])
    write_jsonl(smoke5_input_jsonl, [strip_gold(row) for row in smoke5])
    write_jsonl(smoke5_gold_jsonl, smoke5)


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"
TOOLS_DIR = NOTEBOOK_DIR / "tools"
for path in [PROJECT_ROOT / "src", PROJECT_ROOT, TOOLS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

# Portable bundle preflight: v1.6 depends on v1.5 -> v1.3 -> DocRED helper adapters.
# These files are experiment adapters only; NeoOLAF src itself remains untouched.
REQUIRED_LOCAL_HELPERS = [
    "eventstoryline_native_ablation_v1_6.py",
    "eventstoryline_native_ablation_v1_5.py",
    "eventstoryline_native_ablation_v1_3.py",
    "docred_native_ablation.py",
    "docred_native_ablation_v3.py",
    "docred_native_ablation_v4.py",
]
missing_local_helpers = [name for name in REQUIRED_LOCAL_HELPERS if not (TOOLS_DIR / name).is_file()]
if missing_local_helpers:
    raise FileNotFoundError(
        "Incomplete portable EventStoryLine bundle. Missing helper files under "
        f"{TOOLS_DIR}:\n" + "\n".join(missing_local_helpers)
    )

from eventstoryline_native_ablation_v1_6 import (
    RELATION_IDS,
    analyze_run,
    gold_event_index,
    indexed_token_table,
    load_layer_states,
    project_event_label,
    read_json,
    read_jsonl,
    run_native_pipeline,
    seed_ontology_summary,
    resolve_source_centric_cache_dir,
)

print("PROJECT_ROOT =", PROJECT_ROOT)


PROJECT_ROOT = C:\Users\galencarmedeiro\NeoOLAF


c:\Users\galencarmedeiro\NeoOLAF\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
c:\Users\galencarmedeiro\NeoOLAF\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

In [2]:
# New shared RAGTree preprocessed dataset directory.
# The environment variable remains the safest override if the repository moves again.
RAGTREE_PREPROCESSED_DIR = first_existing_dir(
    "RAGTREE_PREPROCESSED_DIR",
    [
        Path(r"C:\Users\galencarmedeiro\RAGTree\preprocessed"),
        PROJECT_ROOT.parent / "RAGTree/preprocessed",
        PROJECT_ROOT.parent / "ragtree/preprocessed",
        PROJECT_ROOT / "../RAGTree/preprocessed",
        PROJECT_ROOT / "../ragtree/preprocessed",
    ],
)

DATASET_PATHS = {
    "CausalBank": RAGTREE_PREPROCESSED_DIR / "causalbank.jsonl",
    "DocRED": RAGTREE_PREPROCESSED_DIR / "docred_causal.jsonl",
    "EventStoryLine": RAGTREE_PREPROCESSED_DIR / "eventstoryline.jsonl",
    "FinCausal": RAGTREE_PREPROCESSED_DIR / "fincausal.jsonl",
    "MAVEN-ERE": RAGTREE_PREPROCESSED_DIR / "maven_ere.jsonl",
}
EVENTSTORYLINE_DATASET_JSONL = DATASET_PATHS["EventStoryLine"]
SELECTED_DOCUMENT_ID = "EventStoryLine - 1_10ecbplus"

# Same seed ontology used by the RAGTree EventStoryLine experiments.
# Override explicitly with EVENTSTORYLINE_ONTOLOGY_PATH when needed.
ONTOLOGY_PATH = first_existing_path(
    "EVENTSTORYLINE_ONTOLOGY_PATH",
    [
        PROJECT_ROOT.parent / "RAGTree/data/ontology/OWLTime/time.ttl",
        PROJECT_ROOT.parent / "ragtree/data/ontology/OWLTime/time.ttl",
        Path(r"C:\Users\galencarmedeiro\Documents\git\postdoc\RAGTree\data\ontology\OWLTime\time.ttl"),
        PROJECT_ROOT / "../RAGTree/data/ontology/OWLTime/time.ttl",
        PROJECT_ROOT / "../ragtree/data/ontology/OWLTime/time.ttl",
    ],
)

# Controlled normalized benchmark relation schema. This is not the seed ontology.
RELATION_CATALOG = NOTEBOOK_DIR / "ontology/eventstoryline_relation_catalog.json"
RELATION_ALIASES = NOTEBOOK_DIR / "ontology/eventstoryline_relation_aliases.json"
PROFILE_PATH = NOTEBOOK_DIR / "configs/eventstoryline_profile_native_ablation_v1_6.json"
GUIDANCE_PATH = NOTEBOOK_DIR / "configs/guidance_eventstoryline_native_ablation_v1_6.json"
TASK_GUIDANCE_PATH = NOTEBOOK_DIR / "configs/eventstoryline_task_guidance_v1_6.json"

RUNS_ROOT = NOTEBOOK_DIR / "runs/eventstoryline_native_layer_ablation"
RUN_DIR = RUNS_ROOT / "document_1_10ecbplus_v1_6_3_owltime_source_centric_shortcache"
PREPARED_DIR = RUNS_ROOT / "_prepared_eventstoryline_v1_6_3"
INPUT_JSONL = PREPARED_DIR / "eventstoryline_one_input_v1_6_3.jsonl"
GOLD_JSONL = PREPARED_DIR / "eventstoryline_one_gold_v1_6_3.jsonl"
SMOKE5_INPUT_JSONL = PREPARED_DIR / "eventstoryline_smoke5_input_v1_6_3.jsonl"
SMOKE5_GOLD_JSONL = PREPARED_DIR / "eventstoryline_smoke5_gold_v1_6_3.jsonl"

OPENROUTER_HOST = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-oss-20b"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")

# Fixed-source relation requests are independent and run concurrently in Layer 2.
WORKERS = 16
REASONING_EFFORT = "minimal"
RUN_PIPELINE = True
CLEAN_RUN_DIR = True

# v1.6.3: short, portable Layer-2 cache. Override only when desired.
LAYER2_CACHE_DIR = resolve_source_centric_cache_dir()

print("RAGTree preprocessed root:", RAGTREE_PREPROCESSED_DIR)
for dataset_name, dataset_path in DATASET_PATHS.items():
    print(f"{dataset_name:14s} -> {dataset_path}")
print("Selected EventStoryLine document:", SELECTED_DOCUMENT_ID)
print("OWL-Time seed ontology:", ONTOLOGY_PATH)
print("Run dir:", RUN_DIR)
print("Model:", MODEL_NAME)
print("API key available:", bool(API_KEY))
print("Layer-2 short cache:", LAYER2_CACHE_DIR)
print("Layer-2 cache path chars:", len(str(LAYER2_CACHE_DIR)))


RAGTree preprocessed root: C:\Users\galencarmedeiro\RAGTree\preprocessed
CausalBank     -> C:\Users\galencarmedeiro\RAGTree\preprocessed\causalbank.jsonl
DocRED         -> C:\Users\galencarmedeiro\RAGTree\preprocessed\docred_causal.jsonl
EventStoryLine -> C:\Users\galencarmedeiro\RAGTree\preprocessed\eventstoryline.jsonl
FinCausal      -> C:\Users\galencarmedeiro\RAGTree\preprocessed\fincausal.jsonl
MAVEN-ERE      -> C:\Users\galencarmedeiro\RAGTree\preprocessed\maven_ere.jsonl
Selected EventStoryLine document: EventStoryLine - 1_10ecbplus
OWL-Time seed ontology: C:\Users\galencarmedeiro\RAGTree\data\ontology\OWLTime\time.ttl
Run dir: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\eventstoryline_native_layer_ablation\document_1_10ecbplus_v1_6_3_owltime_source_centric_shortcache
Model: openai/gpt-oss-20b
API key available: True
Layer-2 short cache: C:\Users\GALENC~1\AppData\Local\Temp\neoolaf_esl16
Layer-2 cache path chars: 50


## 2. Preflight: stream the real EventStoryLine JSONL, strip gold, and validate the relation-focused configuration


In [3]:

# v1.6.3 Windows-path/cache preflight. Cache is optional for correctness, but
# verify the chosen short directory when the filesystem allows it.
cache_probe_ok = False
cache_probe_error = None
try:
    LAYER2_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    probe = LAYER2_CACHE_DIR / "probe.json"
    probe.write_text('{"ok": true}', encoding="utf-8")
    probe.unlink(missing_ok=True)
    cache_probe_ok = True
except OSError as exc:
    cache_probe_error = f"{type(exc).__name__}: {exc}"
print("Layer-2 cache writable:", cache_probe_ok)
if cache_probe_error:
    print("Layer-2 cache warning (non-fatal in v1.6.3):", cache_probe_error)
assert len((LAYER2_CACHE_DIR / ("a" * 24 + ".json")).__str__()) < 180 or os.name != "nt", (
    "Layer-2 cache path is unexpectedly long on Windows; set "
    "NEOOLAF_EVENTSTORYLINE_CACHE_DIR to a shorter directory."
)

missing_datasets = [str(path) for path in DATASET_PATHS.values() if not path.is_file()]
if missing_datasets:
    raise FileNotFoundError(
        "Missing one or more preprocessed datasets under "
        f"{RAGTREE_PREPROCESSED_DIR}:\n" + "\n".join(missing_datasets)
    )

# Stream the real EventStoryLine JSONL and materialize only the selected document + first 5.
# Gold fields are stripped before the pipeline input is written.
prepare_eventstoryline_views(
    dataset_jsonl=EVENTSTORYLINE_DATASET_JSONL,
    selected_document_id=SELECTED_DOCUMENT_ID,
    input_jsonl=INPUT_JSONL,
    gold_jsonl=GOLD_JSONL,
    smoke5_input_jsonl=SMOKE5_INPUT_JSONL,
    smoke5_gold_jsonl=SMOKE5_GOLD_JSONL,
)

required = [
    INPUT_JSONL, GOLD_JSONL, SMOKE5_INPUT_JSONL, SMOKE5_GOLD_JSONL,
    ONTOLOGY_PATH, RELATION_CATALOG, RELATION_ALIASES,
    PROFILE_PATH, GUIDANCE_PATH, TASK_GUIDANCE_PATH,
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

input_rows = read_jsonl(INPUT_JSONL)
gold_rows = read_jsonl(GOLD_JSONL)
smoke_input_rows = read_jsonl(SMOKE5_INPUT_JSONL)
smoke_gold_rows = read_jsonl(SMOKE5_GOLD_JSONL)
assert len(input_rows) == 1 and len(gold_rows) == 1
assert len(smoke_input_rows) == 5 and len(smoke_gold_rows) == 5
assert "entities" not in input_rows[0] and "relations" not in input_rows[0]
assert all("entities" not in row and "relations" not in row for row in smoke_input_rows)
assert input_rows[0]["document_id"] == gold_rows[0]["document_id"]
assert [x["document_id"] for x in smoke_input_rows] == [x["document_id"] for x in smoke_gold_rows]

profile = read_json(PROFILE_PATH)
task = read_json(TASK_GUIDANCE_PATH)
catalog = read_json(RELATION_CATALOG)
gold = gold_rows[0]
seed_summary = seed_ontology_summary(ONTOLOGY_PATH)
l1_cfg = profile["layers"]["layer01_linguistic_expression_extraction"]
l2_cfg = profile["layers"]["layer02_candidate_enrichment"]

print("Full EventStoryLine dataset:", EVENTSTORYLINE_DATASET_JSONL)
print("Document:", input_rows[0]["document_id"], "-", input_rows[0]["title"])
print("Source characters:", len(input_rows[0]["text"]))
print("Sentences:", len(input_rows[0]["sentences"]))
print("Source tokens:", sum(len(x) for x in input_rows[0]["tokens"]))
print("Gold events (not exposed to pipeline):", len(gold["entities"]))
print("Gold evaluated relations:", sum(len(v) for k, v in gold["relations"].items() if k in RELATION_IDS))
print("Ignored null pairs:", len(gold["relations"].get("null", [])))
print("OWL-Time classes loaded:", seed_summary["class_count"])
print("OWL-Time properties loaded:", seed_summary["property_count"])
print("Controlled task relations:", catalog["property_count"])
print("Relation IDs:", task["allowed_relation_ids"])
print("Layer 1 sentence workers:", l1_cfg["sentence_workers"])
print("Layer 1 coverage reviews:", l1_cfg["coverage_review_passes"])
print("Pair strategy:", l1_cfg["pair_strategy"])
print("Exhaustive-event threshold:", l1_cfg["exhaustive_event_threshold"])
print("Relation-generation LLM calls:", l1_cfg["relation_generation_llm_calls"])
print("Positive evidence required:", l2_cfg["evidence_required_for_positive"])
print("NONE filtered before Layer 3:", l2_cfg["filter_none_before_layer03"])
print("Global-trigger projection fallback:", profile["benchmark_projection"]["fallback_unique_trigger"])
print("Mention-free relation schemas injected:", len(profile["relations"]["allowed"]))
print("Five-document input IDs:", [row["document_id"] for row in smoke_input_rows])

assert seed_summary["class_count"] > 0 or seed_summary["property_count"] > 0
assert catalog["property_count"] == 2
assert set(task["allowed_relation_ids"]) == set(RELATION_IDS)
assert profile["relations"]["allowed"] == []
assert l1_cfg["relation_generation_llm_calls"] == 0
assert l1_cfg["deterministic_unordered_pair_pool"] is True
assert l2_cfg["filter_none_before_layer03"] is True
assert l2_cfg["evidence_required_for_positive"] is True
assert profile["benchmark_projection"]["fallback_unique_trigger"] is False
assert profile["anti_cheating"]["direct_eventstoryline_extraction"] is False
assert profile["anti_cheating"]["source_event_anchoring"] is False
assert profile["anti_cheating"]["gold_pair_hints"] is False
assert profile["anti_cheating"]["post_run_relation_invention"] is False
assert profile["benchmark_projection"]["gold_available_to_pipeline"] is False
assert profile["benchmark_projection"]["same_seed_ontology_as_ragtree"] is True

# Prompt examples are synthetic and schema-exact; no dataset gold pair is embedded.
assert len(task["source_centric_schema_examples"]) >= 5
assert {x["relation"] for x in task["source_centric_schema_examples"]} >= {
    "PRECONDITION", "FALLING_ACTION", "NONE"
}

display(Markdown("### Indexed source table used by Layer 1"))
print(indexed_token_table(input_rows[0]["sentences"], input_rows[0]["tokens"]))

assert profile["anti_cheating"]["gold_event_lexicon"] is False
assert profile["anti_cheating"]["gold_event_count"] is False
assert profile["anti_cheating"]["gold_relation_count"] is False

print("Atomic span review enabled:", l1_cfg.get("atomic_review_enabled"))
print("Pair-local direct-link mode:", l2_cfg.get("direct_link_only"))
print("Candidate closure enabled:", l2_cfg.get("closure_enabled"))
print("Broad NONE review:", l2_cfg.get("none_review_enabled"))
print("Positive verifier:", l2_cfg.get("positive_verifier_enabled"))
print("Fixed source per request:", l2_cfg.get("fixed_source_per_request"))
print("Target decisions:", l2_cfg.get("target_decisions"))


assert l2_cfg["strategy"] == "eventstoryline_source_centric_directed_target_checklist"
assert l2_cfg["target_decisions"] == ["PRECONDITION", "FALLING_ACTION", "NONE"]
assert l2_cfg["none_review_enabled"] is False
assert l2_cfg["positive_verifier_enabled"] is False
assert l2_cfg["closure_enabled"] is False
print("Layer 2 strategy:", l2_cfg["strategy"])
print("Source workers:", l2_cfg["source_workers"])
print("Missing-target retry:", l2_cfg["missing_target_retry"])
print("Conflict-only adjudication:", l2_cfg["conflict_adjudication_enabled"])
print("Broad NONE review:", l2_cfg["none_review_enabled"])
print("Positive verifier:", l2_cfg["positive_verifier_enabled"])


Layer-2 cache writable: True
Full EventStoryLine dataset: C:\Users\galencarmedeiro\RAGTree\preprocessed\eventstoryline.jsonl
Document: EventStoryLine - 1_10ecbplus - 1_10ecbplus
Source characters: 745
Sentences: 6
Source tokens: 147
Gold events (not exposed to pipeline): 15
Gold evaluated relations: 20
Ignored null pairs: 1
OWL-Time classes loaded: 23
OWL-Time properties loaded: 58
Controlled task relations: 2
Relation IDs: ['PRECONDITION', 'FALLING_ACTION']
Layer 1 sentence workers: 8
Layer 1 coverage reviews: 3
Pair strategy: adaptive
Exhaustive-event threshold: 25
Relation-generation LLM calls: 0
Positive evidence required: True
NONE filtered before Layer 3: True
Global-trigger projection fallback: False
Mention-free relation schemas injected: 0
Five-document input IDs: ['EventStoryLine - 1_10ecbplus', 'EventStoryLine - 1_11ecbplus', 'EventStoryLine - 1_12ecbplus', 'EventStoryLine - 1_13ecbplus', 'EventStoryLine - 1_14ecbplus']


### Indexed source table used by Layer 1

[S0] http : / / articles . latimes . com / 2013 / may / 03 / local / la - me - 0504 - lohan - rehab - 20130504
TOKENS 0=http 1=: 2=/ 3=/ 4=articles 5=. 6=latimes 7=. 8=com 9=/ 10=2013 11=/ 12=may 13=/ 14=03 15=/ 16=local 17=/ 18=la 19=- 20=me 21=- 22=0504 23=- 24=lohan 25=- 26=rehab 27=- 28=20130504

[S1] Lindsay Lohan checks into Betty Ford Center
TOKENS 0=Lindsay 1=Lohan 2=checks 3=into 4=Betty 5=Ford 6=Center

[S2] May 03 , 2013
TOKENS 0=May 1=03 2=, 3=2013

[S3] After skipping out on entering a Newport Beach rehabilitation facility and facing the prospect of arrest for violating her probation , Lindsay Lohan has checked into the Betty Ford Center to begin a 90 - day court - mandated stay in her reckless driving conviction .
TOKENS 0=After 1=skipping 2=out 3=on 4=entering 5=a 6=Newport 7=Beach 8=rehabilitation 9=facility 10=and 11=facing 12=the 13=prospect 14=of 15=arrest 16=for 17=violating 18=her 19=probation 20=, 21=Lindsay 22=Lohan 23=has 24=checked 25=into 26=the 27=Betty 28=Fo

## 3. Run the full native Layer 0--12 pipeline

In [4]:
if RUN_PIPELINE:
    if not API_KEY:
        API_KEY = getpass("OpenRouter API key: ").strip().strip('"').strip("'")
    if not API_KEY:
        raise RuntimeError("No OpenRouter API key was provided.")

    final_state = run_native_pipeline(
        project_root=PROJECT_ROOT,
        input_jsonl=INPUT_JSONL,
        ontology_path=ONTOLOGY_PATH,
        profile_path=PROFILE_PATH,
        guidance_path=GUIDANCE_PATH,
        task_guidance_path=TASK_GUIDANCE_PATH,
        relation_catalog_path=RELATION_CATALOG,
        relation_aliases_path=RELATION_ALIASES,
        run_dir=RUN_DIR,
        model_name=MODEL_NAME,
        api_key=API_KEY,
        host=OPENROUTER_HOST,
        workers=WORKERS,
        reasoning_effort=REASONING_EFFORT,
        verbose=True,
        clean_run_dir=CLEAN_RUN_DIR,
    )
    print("Full native run completed.")
    print("Layer-2 cache used:", resolve_source_centric_cache_dir())
else:
    print("RUN_PIPELINE=False: reusing", RUN_DIR)

[NeoOLAF] Run directory: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\eventstoryline_native_layer_ablation\document_1_10ecbplus_v1_6_3_owltime_source_centric_shortcache
[NeoOLAF] from_layer=0, to_layer=None, skip_layers=None
[NeoOLAF] Pipeline has 13 layers
[NeoOLAF] Selected layers: ['layer00_preprocessing', 'layer01_linguistic_expression_extraction', 'layer02_candidate_enrichment', 'layer03_candidate_typing_resolution', 'layer04_candidate_relation_extraction', 'layer05_candidate_triple_generation', 'layer06_concept_relation_induction', 'layer07_hierarchisation', 'layer08_axiom_schemata_extraction', 'layer09_general_axiom_extraction', 'layer10_validation_reasoning', 'layer11_inference_completion', 'layer12_serialization']
[NeoOLAF] Layer 0/12: layer00_preprocessing

[NeoOLAF] Starting layer: layer00_preprocessing
[NeoOLAF] Finished layer: layer00_preprocessing in 0.00s
[NeoOLAF] Layer 1/12: layer01_linguistic_expression_extraction

[NeoOLAF] Starting layer: layer01_l

[NeoOLAF] Finished layer: layer03_candidate_typing_resolution in 0.09s
[NeoOLAF] Layer 4/12: layer04_candidate_relation_extraction

[NeoOLAF] Starting layer: layer04_candidate_relation_extraction
[NeoOLAF][Layer 4] strategy=structured_exact_then_native_parallel_fallback; parallel_workers=8; attempts=1
[NeoOLAF] Finished layer: layer04_candidate_relation_extraction in 0.06s
[NeoOLAF] Layer 5/12: layer05_candidate_triple_generation

[NeoOLAF] Starting layer: layer05_candidate_triple_generation


[NeoOLAF] Finished layer: layer05_candidate_triple_generation in 0.00s
[NeoOLAF] Layer 6/12: layer06_concept_relation_induction

[NeoOLAF] Starting layer: layer06_concept_relation_induction
[NeoOLAF][Layer 6] deterministic ontology-aware concept induction for 19 node candidates; no LLM calls.
[NeoOLAF][Layer 6] deterministic ontology-aware relation induction for 79 relation candidates; no LLM calls.
[NeoOLAF] Finished layer: layer06_concept_relation_induction in 0.01s
[NeoOLAF] Layer 7/12: layer07_hierarchisation

[NeoOLAF] Starting layer: layer07_hierarchisation
[NeoOLAF] Finished layer: layer07_hierarchisation in 0.00s
[NeoOLAF] Layer 8/12: layer08_axiom_schemata_extraction

[NeoOLAF] Starting layer: layer08_axiom_schemata_extraction
[NeoOLAF][Layer 8] strategy=ontology_aware_axiom_schema_generation
[NeoOLAF] Finished layer: layer08_axiom_schemata_extraction in 0.00s
[NeoOLAF] Layer 9/12: layer09_general_axiom_extraction

[NeoOLAF] Starting layer: layer09_general_axiom_extraction
[Ne

[NeoOLAF] Finished layer: layer10_validation_reasoning in 0.01s
[NeoOLAF] Layer 11/12: layer11_inference_completion

[NeoOLAF] Starting layer: layer11_inference_completion
[NeoOLAF][Layer 11] strategy=ontology_aware_semantic_completion
[NeoOLAF][Layer 11] deterministic completion; max_concurrency=16; no LLM calls.
[NeoOLAF] Finished layer: layer11_inference_completion in 0.01s
[NeoOLAF] Layer 12/12: layer12_serialization

[NeoOLAF] Starting layer: layer12_serialization
[NeoOLAF] Exports written to: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\eventstoryline_native_layer_ablation\document_1_10ecbplus_v1_6_3_owltime_source_centric_shortcache\exports
[NeoOLAF] Finished layer: layer12_serialization in 0.19s
[NeoOLAF] Pipeline finished in 242.11s
[NeoOLAF] Saved checkpoint: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\eventstoryline_native_layer_ablation\document_1_10ecbplus_v1_6_3_owltime_source_centric_shortcache\checkpoints\after_selected_pipeline.pkl.

### Runtime evidence saved

- `run_manifest.json` records OWL-Time and the v1.6.2 operational/scientific fingerprint.
- `run_logs/layer01_event_inventory.json` records all accepted/rejected atomic event spans.
- `run_logs/layer01_pair_pool.json` records the deterministic unordered candidate pool.
- `run_logs/layer02_source_centric_decisions.json` records every directed source-to-target decision.
- `run_logs/layer02_source_centric_call_audit.json` records one primary request per fixed source plus focused missing-target retries.
- `run_logs/layer02_source_parse_audit.json` records normalization and evidence-format repairs.
- `run_logs/layer02_direction_conflicts.json` records only opposite-direction positive conflicts and their resolution.
- `run_logs/layer02_relation_decisions.json` records the final one-per-unordered-pair decisions entering native NeoOLAF.
- `run_logs/layer02_batch_cache/` stores resumable prompt-fingerprint responses.
- `NONE` never propagates to Layer 3; no broad false-`NONE` review, positive verifier, or candidate closure is used.
- Layer 2/4 decisions, ontology retrieval, API responses, and all Layer 0–12 states remain saved.


## 4. Projected benchmark metrics and strict native span metrics


In [5]:
summary = analyze_run(
    run_dir=RUN_DIR,
    gold_jsonl=GOLD_JSONL,
    catalog_path=RELATION_CATALOG,
    aliases_path=RELATION_ALIASES,
)

display(pd.DataFrame(summary["layer_summary"]))

print("Projected benchmark relation evaluation; exact-span projection, null excluded")
display(pd.DataFrame([summary["strict_relation_evaluation"]]))

print("Strict native span relation evaluation; unmapped native predictions count as false positives")
display(pd.DataFrame([summary["native_span_relation_evaluation"]]))

print("Projected event-ID evaluation")
display(pd.DataFrame([summary["event_entity_evaluation"]]))

print("Strict native event-span evaluation")
display(pd.DataFrame([summary["native_span_event_evaluation"]]))

print("Relation-endpoint projected event evaluation")
display(pd.DataFrame([summary["relation_endpoint_evaluation"]]))

print("Relation candidate-pool coverage")
display(pd.DataFrame([summary["candidate_pool_evaluation"]]))

print("Per-relation projected metrics")
display(pd.DataFrame(summary["per_relation_metrics"]))

print("Direction/class confusion matrix")
display(pd.DataFrame(summary["relation_confusion_matrix"]))

print("Cumulative evaluation")
display(pd.DataFrame(summary["cumulative_evaluation"]))

print("First-failure counts")
print(summary["failure_counts"])


,layer,layer_name,linguistic_expressions,enriched_expressions,entity_candidates,relation_candidates,attribute_candidates,event_candidates,candidate_relation_assertions,candidate_triples,concept_candidates,ontology_relation_candidates,concept_hierarchy_links,relation_hierarchy_links,axiom_schema_candidates,general_axiom_candidates,completion_candidates,validation_issues,reasoning_inferred_triples
0,0,layer00_preprocessing,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,layer01_linguistic_expression_extraction,190,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2,layer02_candidate_enrichment,190,98,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,3,layer03_candidate_typing_resolution,190,98,0,79,0,19,0,0,0,0,0,0,0,0,0,0,0
4,4,layer04_candidate_relation_extraction,190,98,0,79,0,19,79,0,0,0,0,0,0,0,0,0,0
5,5,layer05_candidate_triple_generation,190,98,0,79,0,19,79,79,0,0,0,0,0,0,0,0,0
6,6,layer06_concept_relation_induction,190,98,0,79,0,19,79,79,0,2,0,0,0,0,0,0,0
7,7,layer07_hierarchisation,190,98,0,79,0,19,79,79,0,2,0,2,0,0,0,0,0
8,8,layer08_axiom_schemata_extraction,190,98,0,79,0,19,79,79,0,2,0,2,6,0,0,0,0
9,9,layer09_general_axiom_extraction,190,98,0,79,0,19,79,79,0,2,0,2,6,8,0,0,0


Projected benchmark relation evaluation; exact-span projection, null excluded


,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
0,45,20,3,42,17,0.066667,0.15,0.092308


Strict native span relation evaluation; unmapped native predictions count as false positives


,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
0,79,20,3,76,17,0.037975,0.15,0.060606


...

Direction/class confusion matrix


,gold_relation,PRECONDITION,FALLING_ACTION,NONE,REVERSED_PRECONDITION,REVERSED_FALLING_ACTION,PAIR_MISSING,INVALID
0,PRECONDITION,3,1,3,0,0,0,0
1,FALLING_ACTION,4,0,2,7,0,0,0


Cumulative evaluation


,layer,layer_name,projected_relation_predicted,projected_relation_gold,projected_relation_true_positive,projected_relation_false_positive,projected_relation_false_negative,projected_relation_precision,projected_relation_recall,projected_relation_f1,...,projected_event_recall,projected_event_f1,native_span_event_predicted,native_span_event_gold,native_span_event_true_positive,native_span_event_false_positive,native_span_event_false_negative,native_span_event_precision,native_span_event_recall,native_span_event_f1
0,0,layer00_preprocessing,0,20,0,0,20,0.000000,0.00,0.000000,...,0.000000,0.000000,0,15,0,0,15,0.000000,0.000000,0.000000
1,1,layer01_linguistic_expression_extraction,0,20,0,0,20,0.000000,0.00,0.000000,...,0.933333,0.965517,19,15,14,5,1,0.736842,0.933333,0.823529
2,2,layer02_candidate_enrichment,0,20,0,0,20,0.000000,0.00,0.000000,...,0.933333,0.965517,19,15,14,5,1,0.736842,0.933333,0.823529
3,3,layer03_candidate_typing_resolution,0,20,0,0,20,0.000000,0.00,0.000000,...,0.933333,0.965517,19,15,14,5,1,0.736842,0.933333,0.823529
4,4,layer04_candidate_relation_extraction,45,20,3,42,17,0.066667,0.15,0.092308,...,0.933333,0.965517,19,15,14,5,1,0.736842,0.933333,0.823529
5,5,layer05_candidate_triple_generation,45,20,3,42,17,0.066667,0.15,0.092308,...,0.933333,0.965517,19,15,14,5,1,0.736842,0.933333,0.823529
6,6,layer06_concept_relation_induction,45,20,3,42,17,0.066667,0.15,0.092308,...,0.933333,0.965517,19,15,14,5,1,0.736842,0.933333,0.823529
7,7,layer07_hierarchisation,45,20,3,42,17,0.066667,0.15,0.092308,...,0.933333,0.965517,19,15,14,5,1,0.736842,0.933333,0.823529
8,8,layer08_axiom_schemata_extraction,45,20,3,42,17,0.066667,0.15,0.092308,...,0.933333,0.965517,19,15,14,5,1,0.736842,0.933333,0.823529
9,9,layer09_general_axiom_extraction,45,20,3,42,17,0.066667,0.15,0.092308,...,0.933333,0.965517,19,15,14,5,1,0.736842,0.933333,0.823529


First-failure counts
{'wrong_direction': 7, 'survived_to_layer05': 3, 'classified_none': 5, 'wrong_relation_class': 5}


## 5. Layer 1 validated event inventory and deterministic unordered relation-pair pool


In [6]:
layer1_rows = read_json(RUN_DIR / "run_logs/layer01_event_relation_instances.json")
layer1_df = pd.DataFrame(layer1_rows)
display(layer1_df)

accepted = layer1_df[layer1_df["status"] == "accepted"] if not layer1_df.empty else layer1_df
if not accepted.empty:
    print("Accepted event mentions:", int((accepted["label"] == "event_mention").sum()))
    print("Deterministic pair expressions:", int((accepted["label"] == "relation_instance").sum()))
    print("Rejected rows:", int((layer1_df["status"] == "rejected").sum()))


,phase,status,expr_id,text,label,pair_id,candidate_reasons
0,atomic_materialization,accepted,expr_00000,S1[2:4]::checks into,event_mention,NaN,NaN
1,atomic_materialization,accepted,expr_00001,S3[1:2]::skipping,event_mention,NaN,NaN
2,atomic_materialization,accepted,expr_00002,S3[4:5]::entering,event_mention,NaN,NaN
3,atomic_materialization,accepted,expr_00003,S3[11:12]::facing,event_mention,NaN,NaN
4,atomic_materialization,accepted,expr_00004,S3[15:16]::arrest,event_mention,NaN,NaN
...,...,...,...,...,...,...,...
185,atomic_pair_pool_materialization,accepted,expr_00185,S5[9:12]::rear - ended || potentially related ...,relation_instance,P00166,[exhaustive_unordered_pair]
186,atomic_pair_pool_materialization,accepted,expr_00186,S5[9:12]::rear - ended || potentially related ...,relation_instance,P00167,[exhaustive_unordered_pair]
187,atomic_pair_pool_materialization,accepted,expr_00187,S5[27:28]::lied || potentially related || S5[3...,relation_instance,P00168,[exhaustive_unordered_pair]
188,atomic_pair_pool_materialization,accepted,expr_00188,S5[27:28]::lied || potentially related || S5[3...,relation_instance,P00169,[exhaustive_unordered_pair]


Accepted event mentions: 19
Deterministic pair expressions: 171
Rejected rows: 0


### Layer 1A span validation/repair and Layer 1 deterministic pair audit


In [7]:
inventory_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_event_inventory.json"))
atomic_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_atomic_span_review.json"))
pair_pool = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_pair_pool.json"))
call_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer01_call_audit.json"))

print("Final atomic event inventory audit")
display(inventory_audit)
print("Atomic KEEP/DROP/REPLACE/SPLIT audit")
display(atomic_audit)

print("Deterministic unordered pair pool rebuilt after atomic refinement")
display(pair_pool)
if not pair_pool.empty:
    print("Initial pair candidates:", len(pair_pool))
    print("Pair strategy values:", sorted(pair_pool["strategy"].dropna().unique()))

print("Layer 1 sentence/review/atomic/pair construction audit")
display(call_audit)
if not call_audit.empty:
    display(call_audit.groupby(["phase", "status"], dropna=False).size().reset_index(name="calls"))


Final atomic event inventory audit


,phase,original_event_key,result_event_key,action,reasons,event_key,reason,status
0,deterministic_atomic_minimization,S1[2:4]::checks into,S1[2:4]::checks into,KEEP,[],NaN,NaN,NaN
1,deterministic_atomic_minimization,S3[1:2]::skipping,S3[1:2]::skipping,KEEP,[],NaN,NaN,NaN
2,deterministic_atomic_minimization,S3[4:5]::entering,S3[4:5]::entering,KEEP,[],NaN,NaN,NaN
3,deterministic_atomic_minimization,S3[11:12]::facing,S3[11:12]::facing,KEEP,[],NaN,NaN,NaN
4,deterministic_atomic_minimization,S3[15:16]::arrest,S3[15:16]::arrest,KEEP,[],NaN,NaN,NaN
5,deterministic_atomic_minimization,S3[17:18]::violating,S3[17:18]::violating,KEEP,[],NaN,NaN,NaN
6,deterministic_atomic_minimization,S3[24:26]::checked into,S3[24:26]::checked into,KEEP,[],NaN,NaN,NaN
7,deterministic_atomic_minimization,S3[31:32]::begin,S3[31:32]::begin,KEEP,[],NaN,NaN,NaN
8,deterministic_atomic_minimization,S3[39:40]::stay,S3[39:40]::stay,KEEP,[],NaN,NaN,NaN
9,deterministic_atomic_minimization,S3[44:45]::conviction,S3[44:45]::conviction,KEEP,[],NaN,NaN,NaN


Atomic KEEP/DROP/REPLACE/SPLIT audit


,phase,original_event_key,result_event_key,action,reasons,event_key,reason,status
0,deterministic_atomic_minimization,S1[2:4]::checks into,S1[2:4]::checks into,KEEP,[],NaN,NaN,NaN
1,deterministic_atomic_minimization,S3[1:2]::skipping,S3[1:2]::skipping,KEEP,[],NaN,NaN,NaN
2,deterministic_atomic_minimization,S3[4:5]::entering,S3[4:5]::entering,KEEP,[],NaN,NaN,NaN
3,deterministic_atomic_minimization,S3[11:12]::facing,S3[11:12]::facing,KEEP,[],NaN,NaN,NaN
4,deterministic_atomic_minimization,S3[15:16]::arrest,S3[15:16]::arrest,KEEP,[],NaN,NaN,NaN
5,deterministic_atomic_minimization,S3[17:18]::violating,S3[17:18]::violating,KEEP,[],NaN,NaN,NaN
6,deterministic_atomic_minimization,S3[24:26]::checked into,S3[24:26]::checked into,KEEP,[],NaN,NaN,NaN
7,deterministic_atomic_minimization,S3[31:32]::begin,S3[31:32]::begin,KEEP,[],NaN,NaN,NaN
8,deterministic_atomic_minimization,S3[39:40]::stay,S3[39:40]::stay,KEEP,[],NaN,NaN,NaN
9,deterministic_atomic_minimization,S3[44:45]::conviction,S3[44:45]::conviction,KEEP,[],NaN,NaN,NaN


Deterministic unordered pair pool rebuilt after atomic refinement


,event_a_id,event_b_id,event_a_key,event_b_key,event_a_sentence,event_b_sentence,candidate_reasons,strategy,phase,pair_id
0,E0000,E0001,S1[2:4]::checks into,S3[1:2]::skipping,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00000
1,E0000,E0002,S1[2:4]::checks into,S3[4:5]::entering,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00001
2,E0000,E0003,S1[2:4]::checks into,S3[11:12]::facing,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00002
3,E0000,E0004,S1[2:4]::checks into,S3[15:16]::arrest,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00003
4,E0000,E0005,S1[2:4]::checks into,S3[17:18]::violating,Lindsay Lohan checks into Betty Ford Center,After skipping out on entering a Newport Beach...,[exhaustive_unordered_pair],exhaustive,initial,P00004
...,...,...,...,...,...,...,...,...,...,...
166,E0015,E0017,S5[9:12]::rear - ended,S5[31:32]::telling,Her latest stay at Betty Ford comes after Loha...,Her latest stay at Betty Ford comes after Loha...,[exhaustive_unordered_pair],exhaustive,initial,P00166
167,E0015,E0018,S5[9:12]::rear - ended,S5[36:37]::driving,Her latest stay at Betty Ford comes after Loha...,Her latest stay at Betty Ford comes after Loha...,[exhaustive_unordered_pair],exhaustive,initial,P00167
168,E0016,E0017,S5[27:28]::lied,S5[31:32]::telling,Her latest stay at Betty Ford comes after Loha...,Her latest stay at Betty Ford comes after Loha...,[exhaustive_unordered_pair],exhaustive,initial,P00168
169,E0016,E0018,S5[27:28]::lied,S5[36:37]::driving,Her latest stay at Betty Ford comes after Loha...,Her latest stay at Betty Ford comes after Loha...,[exhaustive_unordered_pair],exhaustive,initial,P00169


Initial pair candidates: 171
Pair strategy values: ['exhaustive']
Layer 1 sentence/review/atomic/pair construction audit


,phase,sentence_id,status,reason,proposals,new_validated_events,pass_index,inventory_size_after,strategy,event_count,pair_count,exhaustive_event_threshold,input_events,reviewed_events,dropped_events,replacement_or_added_events,final_events
0,sentence_inventory,0.0,skipped,url_sentence,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sentence_inventory,2.0,skipped,date_or_publication_metadata,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,sentence_inventory,1.0,ok,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,sentence_inventory,3.0,ok,NaN,9.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,sentence_inventory,4.0,ok,NaN,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,sentence_inventory,5.0,ok,NaN,5.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,coverage_review,NaN,ok,NaN,1.0,0.0,1.0,18.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,coverage_review,NaN,ok,NaN,1.0,1.0,2.0,19.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,coverage_review,NaN,ok,NaN,3.0,1.0,3.0,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,deterministic_pair_pool,NaN,ok,NaN,NaN,NaN,NaN,NaN,exhaustive,20.0,190.0,25.0,NaN,NaN,NaN,NaN,NaN


,phase,status,calls
0,atomic_span_review,ok,1
1,coverage_review,ok,3
2,deterministic_pair_pool,ok,1
3,deterministic_pair_pool_after_atomic_review,ok,1
4,sentence_inventory,ok,4
5,sentence_inventory,skipped,2


## 6. Source-centric directed relation checklists, focused recovery, and conflict-only adjudication


In [8]:
layer2 = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_relation_decisions.json"))
source_decisions = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_source_centric_decisions.json"))
source_calls = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_source_centric_call_audit.json"))
source_parse = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_source_parse_audit.json"))
prompt_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_compact_prompt_audit.json"))
conflict_audit = read_json(RUN_DIR / "run_logs/layer02_direction_conflicts.json")
closure_pool = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_closure_pair_pool.json"))

print("Final one-per-unordered-pair Layer 2 decisions")
display(layer2)
if not layer2.empty:
    display(layer2.groupby(["status", "decision"], dropna=False).size().reset_index(name="pairs"))
    print("Positive controlled relations entering Layer 3:", int((layer2["status"] == "accepted").sum()))

print("All directed source-to-target decisions")
display(source_decisions)
if not source_decisions.empty:
    display(source_decisions.groupby("relation", dropna=False).size().reset_index(name="directed_decisions"))
    print("Directed target decisions:", len(source_decisions))
    print("Evidence-format repairs:", int(source_decisions.get("evidence_repaired", pd.Series(dtype=bool)).fillna(False).sum()))

print("Fixed-source call and focused recovery audit")
display(source_calls)
if not source_calls.empty:
    display(source_calls.groupby(["phase", "status"], dropna=False).agg(
        calls=("source_event_id", "count"),
        requested_targets=("requested_targets", "sum"),
        resolved_targets=("resolved_targets", "sum"),
        elapsed_seconds=("elapsed_seconds", "sum"),
    ).reset_index())

print("Prompt audit")
display(prompt_audit)
if not prompt_audit.empty:
    source_prompts = prompt_audit[prompt_audit["prompt_kind"] == "source_centric_directed_checklist"]
    print("Primary/recovery source prompts:", len(source_prompts))
    if not source_prompts.empty:
        print("Mean targets per source prompt:", round(source_prompts["target_count"].mean(), 2))
        print("Mean user characters:", round(source_prompts["user_chars"].mean(), 1))
        print("Maximum user characters:", int(source_prompts["user_chars"].max()))

print("Direction conflicts")
print("Conflict count:", conflict_audit.get("conflict_count", 0))
display(pd.DataFrame(conflict_audit.get("audit", [])))

print("Parser/normalization audit")
display(source_parse)
print("Candidate closure pairs (must be zero):", len(closure_pool))
assert len(closure_pool) == 0


Final one-per-unordered-pair Layer 2 decisions


,expr_id,pair_id,phase,event_a,event_b,source,target,selected_relation_id,decision,found,status,evidence_sentence_ids,evidence_text,reason,confidence,ontology_hints
0,expr_00019,P00000,initial,S1[2:4]::checks into,S3[1:2]::skipping,S3[1:2]::skipping,S1[2:4]::checks into,PRECONDITION,B_PRECONDITION_A,True,accepted,"[3, 1]",After skipping out on entering a Newport Beach...,Source-centric E0001: The source sentence expl...,0.95,"[controlled_relation:PRECONDITION, promote_to_..."
1,expr_00020,P00001,initial,S1[2:4]::checks into,S3[4:5]::entering,S3[4:5]::entering,S1[2:4]::checks into,PRECONDITION,B_PRECONDITION_A,True,accepted,"[3, 1]","Source sentence 3 contains ""has checked into t...",Source-centric E0002: Source directly establis...,0.95,"[controlled_relation:PRECONDITION, promote_to_..."
2,expr_00021,P00002,initial,S1[2:4]::checks into,S3[11:12]::facing,S3[11:12]::facing,S1[2:4]::checks into,PRECONDITION,B_PRECONDITION_A,True,accepted,"[3, 1]","Source sentence 3 contains the phrase ""has che...",Source-centric E0003: The source event include...,0.95,"[controlled_relation:PRECONDITION, promote_to_..."
3,expr_00022,P00003,initial,S1[2:4]::checks into,S3[15:16]::arrest,S3[15:16]::arrest,S1[2:4]::checks into,PRECONDITION,B_PRECONDITION_A,True,accepted,"[1, 3]",After skipping out on entering a Newport Beach...,Source-centric E0004: The source sentence expl...,0.95,"[controlled_relation:PRECONDITION, promote_to_..."
4,NaN,P00004,initial,S1[2:4]::checks into,S3[17:18]::violating,NaN,NaN,NaN,NONE,False,filtered_none,[],,Both source-centric directions returned NONE.,0.90,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,expr_00185,P00166,initial,S5[9:12]::rear - ended,S5[31:32]::telling,S5[9:12]::rear - ended,S5[31:32]::telling,PRECONDITION,A_PRECONDITION_B,True,accepted,[5],Her latest stay at Betty Ford comes after Loha...,Source-centric E0015: Source directly states s...,0.95,"[controlled_relation:PRECONDITION, promote_to_..."
167,NaN,P00167,initial,S5[9:12]::rear - ended,S5[36:37]::driving,NaN,NaN,NaN,NONE,False,filtered_none,[],,Both source-centric directions returned NONE.,0.90,NaN
168,expr_00187,P00168,initial,S5[27:28]::lied,S5[31:32]::telling,S5[27:28]::lied,S5[31:32]::telling,PRECONDITION,A_PRECONDITION_B,True,accepted,[5],Her latest stay at Betty Ford comes after Loha...,Conflict adjudication: The lying event is desc...,0.90,"[controlled_relation:PRECONDITION, promote_to_..."
169,NaN,P00169,initial,S5[27:28]::lied,S5[36:37]::driving,NaN,NaN,NaN,NONE,False,filtered_none,[],,Both source-centric directions returned NONE.,0.90,NaN


,status,decision,pairs
0,accepted,A_FALLING_ACTION_B,5
1,accepted,A_PRECONDITION_B,46
2,accepted,B_PRECONDITION_A,28
3,filtered_none,NONE,92


Positive controlled relations entering Layer 3: 79
All directed source-to-target decisions


,source_event_id,source_event_key,target_event_id,target_event_key,relation,evidence_sentence_ids,evidence_text,reason,confidence,phase,evidence_repaired,status
0,E0000,S1[2:4]::checks into,E0001,S3[1:2]::skipping,NONE,[],,Skipping occurs before the check‑in; no direct...,0.9,primary_source_checklist,False,accepted
1,E0000,S1[2:4]::checks into,E0002,S3[4:5]::entering,NONE,[],,Entering a Newport Beach facility is described...,0.9,primary_source_checklist,False,accepted
2,E0000,S1[2:4]::checks into,E0003,S3[11:12]::facing,NONE,[],,Facing arrest is mentioned prior to the check‑...,0.9,primary_source_checklist,False,accepted
3,E0000,S1[2:4]::checks into,E0004,S3[15:16]::arrest,NONE,[],,Arrest is referenced before the check‑in; no d...,0.9,primary_source_checklist,False,accepted
4,E0000,S1[2:4]::checks into,E0005,S3[17:18]::violating,NONE,[],,Violating probation is referenced before the c...,0.9,primary_source_checklist,False,accepted
...,...,...,...,...,...,...,...,...,...,...,...,...
337,E0018,S5[36:37]::driving,E0013,S5[2:3]::stay,NONE,[],,"Source event is a stay; target is stay, no dir...",0.9,primary_source_checklist,False,accepted
338,E0018,S5[36:37]::driving,E0014,S5[6:7]::comes,NONE,[],,"Source event is a stay; target is comes, no di...",0.9,primary_source_checklist,False,accepted
339,E0018,S5[36:37]::driving,E0015,S5[9:12]::rear - ended,NONE,[],,"Source event is a stay; target is rear‑ended, ...",0.9,primary_source_checklist,False,accepted
340,E0018,S5[36:37]::driving,E0016,S5[27:28]::lied,NONE,[],,"Source event is a stay; target is lied, no dir...",0.9,primary_source_checklist,False,accepted


...

Primary/recovery source prompts: 19
Mean targets per source prompt: 18.0
Mean user characters: 13843.0
Maximum user characters: 13843
Direction conflicts
Conflict count: 23


,pair_id,status,decision,attempt,cache_path
0,P00018,ok,NONE,0,C:\Users\GALENC~1\AppData\Local\Temp\neoolaf_e...
1,P00019,ok,NONE,0,C:\Users\GALENC~1\AppData\Local\Temp\neoolaf_e...
2,P00020,ok,NONE,0,C:\Users\GALENC~1\AppData\Local\Temp\neoolaf_e...
3,P00025,ok,NONE,0,C:\Users\GALENC~1\AppData\Local\Temp\neoolaf_e...
4,P00035,ok,NONE,0,C:\Users\GALENC~1\AppData\Local\Temp\neoolaf_e...
5,P00036,ok,NONE,0,C:\Users\GALENC~1\AppData\Local\Temp\neoolaf_e...
6,P00041,ok,NONE,0,C:\Users\GALENC~1\AppData\Local\Temp\neoolaf_e...
7,P00051,ok,NONE,0,C:\Users\GALENC~1\AppData\Local\Temp\neoolaf_e...
8,P00056,ok,A_PRECONDITION_B,0,C:\Users\GALENC~1\AppData\Local\Temp\neoolaf_e...
9,P00070,ok,NONE,0,C:\Users\GALENC~1\AppData\Local\Temp\neoolaf_e...


Parser/normalization audit


,source_event_id,source_event_key,target_event_id,target_event_key,relation,evidence_sentence_ids,evidence_text,reason,confidence,phase,evidence_repaired,status,raw
0,E0003,S3[11:12]::facing,E0000,S1[2:4]::checks into,PRECONDITION,"[3, 1]","Source sentence 3 contains the phrase ""has che...",The source event includes the action of checki...,0.95,primary_source_checklist,False,parsed,NaN
1,E0003,S3[11:12]::facing,E0001,S3[1:2]::skipping,PRECONDITION,[3],"Source sentence 3 contains ""After skipping out...","The source event includes the skipping action,...",0.95,primary_source_checklist,False,parsed,NaN
2,E0003,S3[11:12]::facing,E0002,S3[4:5]::entering,PRECONDITION,[3],"Source sentence 3 contains ""entering a Newport...","The source event includes the entering action,...",0.95,primary_source_checklist,False,parsed,NaN
3,E0003,S3[11:12]::facing,E0004,S3[15:16]::arrest,PRECONDITION,[3],"Source sentence 3 contains ""facing the prospec...","The source event includes the arrest prospect,...",0.95,primary_source_checklist,False,parsed,NaN
4,E0003,S3[11:12]::facing,E0005,S3[17:18]::violating,PRECONDITION,[3],"Source sentence 3 contains ""violating her prob...",The source event includes the violating action...,0.95,primary_source_checklist,False,parsed,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
338,E0014,S5[6:7]::comes,E0013,S5[2:3]::stay,NONE,[],,The target event is identical to the source ev...,0.90,primary_source_checklist,False,parsed,NaN
339,E0014,S5[6:7]::comes,E0015,S5[9:12]::rear - ended,NONE,[],,The source event references the rear‑ended eve...,0.90,primary_source_checklist,False,parsed,NaN
340,E0014,S5[6:7]::comes,E0016,S5[27:28]::lied,NONE,[],,The source event mentions the lying action but...,0.90,primary_source_checklist,False,parsed,NaN
341,E0014,S5[6:7]::comes,E0017,S5[31:32]::telling,NONE,[],,The source event mentions telling but does not...,0.90,primary_source_checklist,False,parsed,NaN


Candidate closure pairs (must be zero): 0


## 7. Exact gold-relation failure trace and confusion matrix


In [9]:
trace_df = pd.read_csv(RUN_DIR / "analysis/gold_relation_trace.csv")
confusion_df = pd.read_csv(RUN_DIR / "analysis/relation_confusion_matrix.csv")
display(trace_df)
display(
    trace_df.groupby("first_failure", dropna=False)
    .size()
    .reset_index(name="gold_relations")
    .sort_values("gold_relations", ascending=False)
)
display(confusion_df)


,source_key,relation_id,target_key,first_failure,pair_in_pool,decision_status,predicted_decision,predicted_source,predicted_relation,predicted_target
0,S1[2:4]::checks into,FALLING_ACTION,S3[15:16]::arrest,wrong_direction,True,accepted,B_PRECONDITION_A,S3[15:16]::arrest,PRECONDITION,S1[2:4]::checks into
1,S1[2:4]::checks into,FALLING_ACTION,S3[4:5]::entering,wrong_direction,True,accepted,B_PRECONDITION_A,S3[4:5]::entering,PRECONDITION,S1[2:4]::checks into
2,S1[2:4]::checks into,PRECONDITION,S3[39:40]::stay,survived_to_layer05,True,accepted,A_PRECONDITION_B,S1[2:4]::checks into,PRECONDITION,S3[39:40]::stay
3,S1[2:4]::checks into,PRECONDITION,S5[2:3]::stay,classified_none,True,filtered_none,NONE,NaN,NaN,NaN
4,S3[15:16]::arrest,FALLING_ACTION,S3[17:18]::violating,wrong_relation_class,True,accepted,A_PRECONDITION_B,S3[15:16]::arrest,PRECONDITION,S3[17:18]::violating
5,S3[15:16]::arrest,PRECONDITION,S3[24:26]::checked into,survived_to_layer05,True,accepted,A_PRECONDITION_B,S3[15:16]::arrest,PRECONDITION,S3[24:26]::checked into
6,S3[24:26]::checked into,PRECONDITION,S3[39:40]::stay,wrong_relation_class,True,accepted,A_FALLING_ACTION_B,S3[24:26]::checked into,FALLING_ACTION,S3[39:40]::stay
7,S3[24:26]::checked into,PRECONDITION,S5[2:3]::stay,classified_none,True,filtered_none,NONE,NaN,NaN,NaN
8,S3[39:40]::stay,FALLING_ACTION,S3[44:45]::conviction,wrong_direction,True,accepted,B_PRECONDITION_A,S3[44:45]::conviction,PRECONDITION,S3[39:40]::stay
9,S3[39:40]::stay,FALLING_ACTION,S5[27:28]::lied,wrong_direction,True,accepted,B_PRECONDITION_A,S5[27:28]::lied,PRECONDITION,S3[39:40]::stay


,first_failure,gold_relations
2,wrong_direction,7
0,classified_none,5
3,wrong_relation_class,5
1,survived_to_layer05,3


,gold_relation,PRECONDITION,FALLING_ACTION,NONE,REVERSED_PRECONDITION,REVERSED_FALLING_ACTION,PAIR_MISSING,INVALID
0,PRECONDITION,3,1,3,0,0,0,0
1,FALLING_ACTION,4,0,2,7,0,0,0


## 8. Native event candidates, assertions and triples

In [10]:
states = {index: state for index, _, state in load_layer_states(RUN_DIR)}
layer3 = states.get(3)
layer4 = states.get(4)
layer5 = states.get(5)
layer6 = states.get(6)
layer11 = states.get(11)

if layer3:
    print("Layer 3 event candidates")
    display(pd.DataFrame([{
        "candidate_id": c.candidate_id,
        "canonical_label": c.canonical_label,
        "mentions": [m.text for m in c.mentions],
        "ontology_hints": c.ontology_hints,
    } for c in layer3.event_candidates or []]))

    print("Layer 3 relation candidates")
    display(pd.DataFrame([{
        "candidate_id": c.candidate_id,
        "canonical_label": c.canonical_label,
        "mentions": [m.text for m in c.mentions],
        "controlled_hints": [h for h in c.ontology_hints if str(h).lower().startswith("controlled_relation:")],
    } for c in layer3.relation_candidates or []]))
    assert all(c.mentions for c in layer3.relation_candidates or []), "Mention-free relation candidate detected."

if layer4:
    print("Layer 4 assertions")
    display(pd.DataFrame([{
        "source": x.source_candidate_label,
        "predicate": x.relation_label,
        "target": x.target_candidate_label,
        "confidence": x.confidence,
    } for x in layer4.candidate_relation_assertions or []]))

if layer5:
    print("Layer 5 triples")
    display(pd.DataFrame([{
        "subject": x.subject_label,
        "predicate": x.predicate_label,
        "object": x.object_label,
        "confidence": x.confidence,
    } for x in layer5.candidate_triples or []]))

print("Layer 6 ontology relation candidates:", len(layer6.ontology_relation_candidates or []) if layer6 else None)
print("Layer 11 completion candidates:", len(layer11.completion_candidates or []) if layer11 else None)

Layer 3 event candidates


,candidate_id,canonical_label,mentions,ontology_hints
0,cand_s_00000,S1[2:4]::checks into,[S1[2:4]::checks into],"[semantic_role:event_mention, candidate_family..."
1,cand_s_00001,S3[1:2]::skipping,[S3[1:2]::skipping],"[semantic_role:event_mention, candidate_family..."
2,cand_s_00002,S3[4:5]::entering,[S3[4:5]::entering],"[semantic_role:event_mention, candidate_family..."
3,cand_s_00003,S3[11:12]::facing,[S3[11:12]::facing],"[semantic_role:event_mention, candidate_family..."
4,cand_s_00004,S3[15:16]::arrest,[S3[15:16]::arrest],"[semantic_role:event_mention, candidate_family..."
5,cand_s_00005,S3[17:18]::violating,[S3[17:18]::violating],"[semantic_role:event_mention, candidate_family..."
6,cand_s_00006,S3[24:26]::checked into,[S3[24:26]::checked into],"[semantic_role:event_mention, candidate_family..."
7,cand_s_00007,S3[31:32]::begin,[S3[31:32]::begin],"[semantic_role:event_mention, candidate_family..."
8,cand_s_00008,S3[39:40]::stay,[S3[39:40]::stay],"[semantic_role:event_mention, candidate_family..."
9,cand_s_00009,S3[44:45]::conviction,[S3[44:45]::conviction],"[semantic_role:event_mention, candidate_family..."


Layer 3 relation candidates


,candidate_id,canonical_label,mentions,controlled_hints
0,cand_r_00000,PRECONDITION,[S3[1:2]::skipping || PRECONDITION || S1[2:4]:...,[controlled_relation:PRECONDITION]
1,cand_r_00001,PRECONDITION,[S3[4:5]::entering || PRECONDITION || S1[2:4]:...,[controlled_relation:PRECONDITION]
2,cand_r_00002,PRECONDITION,[S3[11:12]::facing || PRECONDITION || S1[2:4]:...,[controlled_relation:PRECONDITION]
3,cand_r_00003,PRECONDITION,[S3[15:16]::arrest || PRECONDITION || S1[2:4]:...,[controlled_relation:PRECONDITION]
4,cand_r_00004,PRECONDITION,[S3[24:26]::checked into || PRECONDITION || S1...,[controlled_relation:PRECONDITION]
...,...,...,...,...
74,cand_r_00074,PRECONDITION,[S5[27:28]::lied || PRECONDITION || S5[6:7]::c...,[controlled_relation:PRECONDITION]
75,cand_r_00075,PRECONDITION,[S5[31:32]::telling || PRECONDITION || S5[6:7]...,[controlled_relation:PRECONDITION]
76,cand_r_00076,PRECONDITION,[S5[9:12]::rear - ended || PRECONDITION || S5[...,[controlled_relation:PRECONDITION]
77,cand_r_00077,PRECONDITION,[S5[9:12]::rear - ended || PRECONDITION || S5[...,[controlled_relation:PRECONDITION]


Layer 4 assertions


,source,predicate,target,confidence
0,S3[1:2]::skipping,PRECONDITION,S1[2:4]::checks into,1.0
1,S3[4:5]::entering,PRECONDITION,S1[2:4]::checks into,1.0
2,S3[11:12]::facing,PRECONDITION,S1[2:4]::checks into,1.0
3,S3[15:16]::arrest,PRECONDITION,S1[2:4]::checks into,1.0
4,S3[24:26]::checked into,PRECONDITION,S1[2:4]::checks into,1.0
...,...,...,...,...
74,S5[27:28]::lied,PRECONDITION,S5[6:7]::comes,1.0
75,S5[31:32]::telling,PRECONDITION,S5[6:7]::comes,1.0
76,S5[9:12]::rear - ended,PRECONDITION,S5[27:28]::lied,1.0
77,S5[9:12]::rear - ended,PRECONDITION,S5[31:32]::telling,1.0


Layer 5 triples


,subject,predicate,object,confidence
0,S3[1:2]::skipping,PRECONDITION,S1[2:4]::checks into,1.0
1,S3[4:5]::entering,PRECONDITION,S1[2:4]::checks into,1.0
2,S3[11:12]::facing,PRECONDITION,S1[2:4]::checks into,1.0
3,S3[15:16]::arrest,PRECONDITION,S1[2:4]::checks into,1.0
4,S3[24:26]::checked into,PRECONDITION,S1[2:4]::checks into,1.0
...,...,...,...,...
74,S5[27:28]::lied,PRECONDITION,S5[6:7]::comes,1.0
75,S5[31:32]::telling,PRECONDITION,S5[6:7]::comes,1.0
76,S5[9:12]::rear - ended,PRECONDITION,S5[27:28]::lied,1.0
77,S5[9:12]::rear - ended,PRECONDITION,S5[31:32]::telling,1.0


Layer 6 ontology relation candidates: 2
Layer 11 completion candidates: 0


## 9. Exact-span-only event projection audit


In [11]:
projection = pd.read_csv(RUN_DIR / "analysis/event_projection_audit.csv")
display(projection)

# Exact-span demonstration using source structure. Gold is consulted only here,
# after the pipeline has completed. Trigger-only fallbacks are disabled.
first_gold_key = next(iter(gold_event_index(gold)["keys_by_id"].values()))[0]
print(first_gold_key)
print(project_event_label(first_gold_key, gold))


,event_id,method,label,candidate_event_ids
0,EVENT_34cccabacad4dea93d3e762114dc05cd,exact_sentence_token_span,S1[2:4]::checks into,NaN
1,NaN,unmapped_or_ambiguous_exact_span,S3[1:2]::skipping,[]
2,EVENT_b79df11f8737f700691af4f8a7132190,exact_sentence_token_span,S3[4:5]::entering,NaN
3,NaN,unmapped_or_ambiguous_exact_span,S3[11:12]::facing,[]
4,EVENT_c332a6b60af0495f34856f112dc68632,exact_sentence_token_span,S3[15:16]::arrest,NaN
5,EVENT_9c07ef70f303f23dd82b32fb28a61ecb,exact_sentence_token_span,S3[17:18]::violating,NaN
6,EVENT_d7aec1e4ff20c3264f9fb7686355eb8a,exact_sentence_token_span,S3[24:26]::checked into,NaN
7,NaN,unmapped_or_ambiguous_exact_span,S3[31:32]::begin,[]
8,EVENT_2ea9a901cc230afcc371bcffee02a551,exact_sentence_token_span,S3[39:40]::stay,NaN
9,EVENT_7d29e127b36f34a94ceb0f13f9ea3a3b,exact_sentence_token_span,S3[44:45]::conviction,NaN


S4[15:16]::violation
{'event_id': 'EVENT_434faad0c9ec71baa6a91df3c385299b', 'method': 'exact_sentence_token_span', 'label': 'S4[15:16]::violation'}


## 10. Speed, concurrency and errors

In [12]:
def optional_jsonl(path: Path) -> pd.DataFrame:
    return pd.DataFrame(read_jsonl(path)) if path.is_file() else pd.DataFrame()

calls = optional_jsonl(RUN_DIR / "run_logs/llm_calls.jsonl")
errors = optional_jsonl(RUN_DIR / "run_logs/llm_errors.jsonl")
parse_errors = optional_jsonl(RUN_DIR / "run_logs/llm_parse_errors.jsonl")
retrieval = optional_jsonl(RUN_DIR / "run_logs/ontology_retrieval.jsonl")

if not calls.empty:
    display(calls)
    display(calls.groupby("layer_tag").agg(
        calls=("call_index", "count"),
        total_recorded_seconds=("elapsed_seconds", "sum"),
        maximum_call_seconds=("elapsed_seconds", "max"),
        mean_system_chars=("system_chars", "mean"),
        mean_user_chars=("user_chars", "mean"),
        mean_response_chars=("response_chars", "mean"),
    ).reset_index())

print("Backend/API errors:", len(errors))
if not errors.empty: display(errors)
print("JSON parse errors:", len(parse_errors))
if not parse_errors.empty: display(parse_errors)
if not retrieval.empty:
    display(retrieval.groupby("layer_name").size().reset_index(name="retrieval_calls"))

manifest = read_json(RUN_DIR / "run_manifest.json")
print("Wall-clock pipeline seconds:", manifest.get("elapsed_seconds"))

,call_index,layer_tag,model,temperature,message_count,system_chars,user_chars,max_tokens,request_timeout,started_at,status,elapsed_seconds,response_chars,response_path,json_parse_ok,parsed_type,parse_error
0,3,layer01_atomic_event_inventory_v1_6,openai/gpt-oss-20b,0.0,2,1675,2500,8192,180,2026-08-18 13:49:09,ok,2.522,471,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None
1,1,layer01_atomic_event_inventory_v1_6,openai/gpt-oss-20b,0.0,2,1675,2281,8192,180,2026-08-18 13:49:09,ok,3.205,206,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None
2,2,layer01_atomic_event_inventory_v1_6,openai/gpt-oss-20b,0.0,2,1675,2841,8192,180,2026-08-18 13:49:09,ok,7.114,1299,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None
3,4,layer01_atomic_event_inventory_v1_6,openai/gpt-oss-20b,0.0,2,1675,2651,8192,180,2026-08-18 13:49:09,ok,15.180,737,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None
4,5,layer01_atomic_event_inventory_v1_6,openai/gpt-oss-20b,0.0,2,986,5738,8192,180,2026-08-18 13:49:25,ok,3.461,208,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None
5,6,layer01_atomic_event_inventory_v1_6,openai/gpt-oss-20b,0.0,2,1016,5738,8192,180,2026-08-18 13:49:28,ok,86.352,207,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None
6,7,layer01_atomic_event_inventory_v1_6,openai/gpt-oss-20b,0.0,2,1027,5780,8192,180,2026-08-18 13:50:54,ok,5.298,488,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None
7,8,layer01_atomic_event_inventory_v1_6,openai/gpt-oss-20b,0.0,2,1098,5048,8192,180,2026-08-18 13:51:00,ok,12.106,1060,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None
8,12,layer02_source_centric_relations_v1_6,openai/gpt-oss-20b,0.0,2,2547,13843,4096,180,2026-08-18 13:51:13,ok,12.193,5031,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None
9,10,layer02_source_centric_relations_v1_6,openai/gpt-oss-20b,0.0,2,2547,13843,4096,180,2026-08-18 13:51:13,ok,16.690,6758,C:\Users\galencarmedeiro\NeoOLAF\examples\RAGT...,True,dict,None


,layer_tag,calls,total_recorded_seconds,maximum_call_seconds,mean_system_chars,mean_user_chars,mean_response_chars
0,layer01_atomic_event_inventory_v1_6,8,135.238,86.352,1353.375000,4072.125000,584.500000
1,layer02_source_centric_relations_v1_6,22,502.463,69.317,2275.227273,15113.454545,4156.454545


Backend/API errors: 1


,call_index,layer_tag,model,temperature,message_count,system_chars,user_chars,max_tokens,request_timeout,started_at,status,elapsed_seconds,error_type,error,traceback
0,24,layer02_source_centric_relations_v1_6,openai/gpt-oss-20b,0.0,2,2547,13843,4096,180,2026-08-18 13:51:41,error,43.679,RuntimeError,No final message.content returned by backend o...,"Traceback (most recent call last):\n File ""C:..."


JSON parse errors: 0
Wall-clock pipeline seconds: 242.18231558799744


## Success checklist before the five-document batch

1. The new RAGTree preprocessed directory and sibling OWL-Time file resolve and remains the external seed ontology.
2. The v1.5 atomic event inventory remains unchanged and retains high exact-span recall.
3. Layer 1 deterministically builds the complete unordered pair pool for this small document with zero relation-generation LLM calls.
4. Layer 2 makes one fixed-source request per event and classifies every candidate target as `PRECONDITION`, `FALLING_ACTION`, or `NONE`.
5. Missing target IDs receive exactly one focused recovery request; resolved positive decisions are never discarded because of evidence formatting.
6. Direction is fixed by each source request; only opposite-direction positive conflicts receive compact adjudication.
7. Broad false-`NONE` review, destructive positive verification, and candidate closure are disabled.
8. `NONE` and malformed decisions are absent from Layer 3 relation candidates.
9. Candidate-pool recall, wrong direction/class, `NONE`, and downstream survival remain separately reported.
10. Gold remains unavailable until post-Layer-12 evaluation. Freeze v1.6.2 only after this one-document relation result is satisfactory.
